In [1]:
import subprocess
import os
import json
from pathlib import Path
import pdfplumber

In [ ]:
import psycopg2
from psycopg2 import sql, extras
import logging


In [ ]:
# Set up basic logging to see success/failure messages
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

In [ ]:
DB_PARAMS = {
    'dbname': 'thecall',
    'user': 'postgres',        
    'password': 'password',    
    'host': 'localhost',           
    'port': '5432'                 
}

In [ ]:
def connect_to_db(params):
    """Establishes a connection to the PostgreSQL database."""
    conn = None
    try:
        logging.info("Attempting to connect to PostgreSQL database...")
        conn = psycopg2.connect(**params)
        # Set autocommit to False so we can control transactions (needed for UPDATE)
        conn.autocommit = False 
        logging.info("Connection successful.")
        return conn
    except psycopg2.Error as e:
        logging.error(f"Error connecting to the database: {e}")
        return None

In [ ]:
def collect_data_from_db(conn, table_schema, table_name):
    try:
        with conn.cursor() as cur:
            # Correctly use sql.Identifier for both schema and table separately
            query = sql.SQL("SELECT id, filename, folder FROM {}.{} WHERE json_raw IS NULL").format(
                sql.Identifier(table_schema),
                sql.Identifier(table_name)
            )
            cur.execute(query)
            return cur.fetchall()
    except psycopg2.Error as e:
        logging.error(f"Error fetching data: {e}")
        return []

In [2]:
def extract_to_standard_json(all_data, target_dir, soffice_path):
    target_root = Path(target_dir).resolve()
    
    # Standardize output folders
    pdf_output_dir = target_root / "generated_pdfs"
    
    for folder in [pdf_output_dir]:
        folder.mkdir(parents=True, exist_ok=True)
    
    for file_path in all_data:
        if file_path.is_dir() or file_path.suffix in [".pdf", ".json"] or file_path.name.startswith(".~lock"):
            continue

        print(f"Processing: {file_path.name}")
        
        try:
            # 1. Convert using the EXPLICIT Draw filter
            # Using 'pdf:draw_pdf_Export' ensures LibreOffice Draw handles the output
            subprocess.run([
                soffice_path, "--headless", 
                "--convert-to", "pdf:draw_pdf_Export",
                "--outdir", str(pdf_output_dir),
                str(file_path)
            ], check=True, env=clean_env, capture_output=True)
            
            # 2. Locate the resulting PDF
            # LibreOffice output name logic: OriginalName.pdf
            pdf_path = pdf_output_dir / f"{file_path.name}.pdf"
            json_path = json_output_dir / f"{file_path.name}.json"

            if not pdf_path.exists():
                print(f"   !! PDF generation failed for {file_path.name}. Check permissions or LibreOffice version.")
                continue

            # 3. Extraction logic
            json_data = {
                "source_file": str(file_path).replace("\\", "/"),
                "document_name": file_path.name,
                "page_count": 0,
                "metadata": {"title": "", "source": "QuarkXPress Legacy"},
                "pages": []
            }

            with pdfplumber.open(pdf_path) as pdf:
                json_data["page_count"] = len(pdf.pages)
                for i, page in enumerate(pdf.pages):
                    page_text = page.extract_text() or ""
                    json_data["pages"].append({
                        "page_name": str(i + 1),
                        "frames": [{
                            "bounds": [0, 0, 0, 0],
                            "paragraphs": [{"style": "Normal", "text": line} for line in page_text.split('\n')],
                            "text": page_text
                        }]
                    })
            
            with open(json_path, "w", encoding="utf-8") as f:
                json.dump(json_data, f, indent=4, ensure_ascii=False)
                
            print(f"   Done -> {json_path.name}")

        except Exception as e:
            print(f"   Failed {file_path.name}: {e}")

In [ ]:
def update_dbs(conn, table_schema, table_name, results):
    if not results:
        logging.info("No results to update.")
        return

    try:
        with conn.cursor() as cur:
            # We add ::json to t.json_content to cast the text to a JSON type
            update_query = """
                UPDATE articles.filelist AS f
                SET json_raw = t.json_content::json
                FROM (VALUES %s) AS t(json_content, filename, folder)
                WHERE f.filename = t.filename AND f.folder = t.folder
            """
            
            extras.execute_values(cur, update_query, results)
            conn.commit()
            logging.info(f"Successfully updated {len(results)} rows in the database.")
            
    except psycopg2.Error as e:
        conn.rollback()
        logging.error(f"Database update failed: {e}")

In [ ]:
def main():
    conn = connect_to_db(DB_PARAMS)

    # INPUT_FOLDER = r"E:/Callproject/assembled"
    OUTPUT_FOLDER = r"E:/Callproject/c_out_quark"
    LIBREOFFICE_BIN = r"C:\Program Files\LibreOffice\program\soffice.exe" 
 
    if conn:
        try:
            table_schema = 'articles'
            table_name = 'filelist'

            # 1. Fetch existing entries
            all_data = collect_data_from_db(conn, table_schema, table_name)
            
            if not all_data:
                logging.warning("No data retrieved from DB. Exiting.")
                return

            # 2 
            results = extract_to_standard_json(all_data, OUTPUT_FOLDER, LIBREOFFICE_BIN)
            
            # 3. Batch update the DB
            update_dbs(conn, table_schema, table_name, results)
            
        finally:
            if conn:
                conn.close()
                logging.info("Database connection closed.")

In [4]:
if __name__ == "__main__":
    main()


Processing: C0000-Licensed Home
   Done -> C0000-Licensed Home.json
Processing: C0520-Child Care avail
   Done -> C0520-Child Care avail.json
